# Linear Regression: A Comprehensive Guide

This notebook provides a complete exploration of Linear Regression, from mathematical foundations to practical implementation and evaluation.

## Table of Contents

1. [Theory Section](#1.-Theory-Section)
2. [Implementation from Scratch](#2.-Implementation-from-Scratch)
3. [Training & Optimization](#3.-Training-&-Optimization)
4. [Diagnostics & Evaluation](#4.-Diagnostics-&-Evaluation)
5. [Visualizations](#5.-Visualizations)
6. [Use Cases & Guidelines](#6.-Use-Cases-&-Guidelines)
7. [Comparison with sklearn](#7.-Comparison-with-sklearn)

---
## 1. Theory Section

### 1.1 Introduction to Linear Regression

Linear regression models the relationship between a dependent variable $y$ and one or more independent variables $X$ by fitting a linear equation:

$$\hat{y} = X\beta + \epsilon$$

Where:
- $\hat{y}$ is the predicted value
- $X$ is the feature matrix (including bias term)
- $\beta$ is the vector of coefficients (weights)
- $\epsilon$ is the error term

### 1.2 Ordinary Least Squares (OLS) Derivation

The goal is to find $\beta$ that minimizes the sum of squared residuals:

$$J(\beta) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 = (y - X\beta)^T(y - X\beta)$$

Expanding:
$$J(\beta) = y^Ty - 2\beta^TX^Ty + \beta^TX^TX\beta$$

Taking the derivative with respect to $\beta$ and setting to zero:
$$\frac{\partial J}{\partial \beta} = -2X^Ty + 2X^TX\beta = 0$$

Solving for $\beta$:
$$X^TX\beta = X^Ty$$
$$\beta = (X^TX)^{-1}X^Ty$$

This is the **Normal Equation**.

### 1.3 The Normal Equation

The closed-form solution:
$$\beta = (X^TX)^{-1}X^Ty$$

**Advantages:**
- Direct solution, no hyperparameters
- Works well for small to medium datasets

**Disadvantages:**
- Computing $(X^TX)^{-1}$ is $O(n^3)$ for $n$ features
- May be numerically unstable if $X^TX$ is singular or near-singular
- Memory intensive for large feature sets

### 1.4 Gradient Descent Approach

For large datasets, iterative optimization via gradient descent is preferred.

**Cost Function:**
$$J(\beta) = \frac{1}{2n}\sum_{i=1}^{n}(h_\beta(x^{(i)}) - y^{(i)})^2 = \frac{1}{2n}||X\beta - y||^2$$

**Gradient:**
$$\nabla J(\beta) = \frac{1}{n}X^T(X\beta - y)$$

**Update Rule:**
$$\beta := \beta - \alpha \nabla J(\beta)$$

Where $\alpha$ is the learning rate.

**Variants:**
- **Batch GD**: Uses all samples per update
- **Stochastic GD (SGD)**: Uses one sample per update
- **Mini-batch GD**: Uses a subset of samples per update

### 1.5 Key Assumptions of Linear Regression

For OLS to produce the Best Linear Unbiased Estimator (BLUE), the following assumptions must hold:

#### 1. Linearity
The relationship between X and y is linear: $E[y|X] = X\beta$

#### 2. Homoscedasticity
Constant variance of errors: $Var(\epsilon_i) = \sigma^2$ for all $i$

#### 3. Independence
Errors are independent: $Cov(\epsilon_i, \epsilon_j) = 0$ for $i \neq j$

#### 4. Normality
Errors are normally distributed: $\epsilon \sim N(0, \sigma^2I)$

#### 5. No Multicollinearity
$X^TX$ is invertible (features are not perfectly correlated)

### 1.6 Time and Space Complexity

| Method | Time Complexity | Space Complexity |
|--------|-----------------|------------------|
| Normal Equation | $O(n \cdot p^2 + p^3)$ | $O(p^2)$ |
| Gradient Descent | $O(k \cdot n \cdot p)$ | $O(p)$ |

Where:
- $n$ = number of samples
- $p$ = number of features
- $k$ = number of iterations (for GD)

**Recommendation:**
- Use Normal Equation when $p < 10,000$
- Use Gradient Descent when $p \geq 10,000$ or data doesn't fit in memory

---
## 2. Implementation from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Literal

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
class LinearRegression:
    """
    Linear Regression implementation with both closed-form and gradient descent solutions.
    
    Parameters
    ----------
    method : str, default='closed_form'
        Optimization method: 'closed_form' (normal equation) or 'gradient_descent'
    learning_rate : float, default=0.01
        Learning rate for gradient descent
    n_iterations : int, default=1000
        Number of iterations for gradient descent
    tolerance : float, default=1e-6
        Convergence tolerance for gradient descent
    fit_intercept : bool, default=True
        Whether to fit an intercept term
    
    Attributes
    ----------
    weights_ : ndarray of shape (n_features,) or (n_features + 1,)
        Coefficient vector (includes intercept if fit_intercept=True)
    cost_history_ : list
        Cost function values during gradient descent training
    """
    
    def __init__(
        self,
        method: Literal['closed_form', 'gradient_descent'] = 'closed_form',
        learning_rate: float = 0.01,
        n_iterations: int = 1000,
        tolerance: float = 1e-6,
        fit_intercept: bool = True
    ):
        self.method = method
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.tolerance = tolerance
        self.fit_intercept = fit_intercept
        
        # Attributes set during fitting
        self.weights_: Optional[np.ndarray] = None
        self.cost_history_: list = []
    
    def _add_intercept(self, X: np.ndarray) -> np.ndarray:
        """Add a column of ones for the intercept term."""
        return np.column_stack([np.ones(X.shape[0]), X])
    
    def _compute_cost(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute the mean squared error cost function."""
        n_samples = X.shape[0]
        predictions = X @ self.weights_
        cost = (1 / (2 * n_samples)) * np.sum((predictions - y) ** 2)
        return cost
    
    def _fit_closed_form(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        Fit using the normal equation: beta = (X'X)^(-1) X'y
        
        Uses numpy's lstsq for numerical stability instead of direct inversion.
        """
        # Use lstsq for better numerical stability than direct inversion
        self.weights_, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    
    def _fit_gradient_descent(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        Fit using batch gradient descent.
        
        Update rule: beta := beta - alpha * (1/n) * X'(X*beta - y)
        """
        n_samples, n_features = X.shape
        
        # Initialize weights to zeros
        self.weights_ = np.zeros(n_features)
        self.cost_history_ = []
        
        for iteration in range(self.n_iterations):
            # Compute predictions
            predictions = X @ self.weights_
            
            # Compute gradient
            gradient = (1 / n_samples) * (X.T @ (predictions - y))
            
            # Update weights
            self.weights_ -= self.learning_rate * gradient
            
            # Record cost
            cost = self._compute_cost(X, y)
            self.cost_history_.append(cost)
            
            # Check convergence
            if iteration > 0:
                cost_change = abs(self.cost_history_[-2] - self.cost_history_[-1])
                if cost_change < self.tolerance:
                    print(f"Converged at iteration {iteration}")
                    break
    
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LinearRegression':
        """
        Fit the linear regression model.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training feature matrix
        y : ndarray of shape (n_samples,)
            Target values
        
        Returns
        -------
        self : LinearRegression
            Fitted estimator
        """
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
        
        # Add intercept column if needed
        if self.fit_intercept:
            X = self._add_intercept(X)
        
        # Fit using selected method
        if self.method == 'closed_form':
            self._fit_closed_form(X, y)
        elif self.method == 'gradient_descent':
            self._fit_gradient_descent(X, y)
        else:
            raise ValueError(f"Unknown method: {self.method}")
        
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Make predictions using the fitted model.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Feature matrix for prediction
        
        Returns
        -------
        y_pred : ndarray of shape (n_samples,)
            Predicted values
        """
        if self.weights_ is None:
            raise RuntimeError("Model must be fitted before making predictions")
        
        X = np.asarray(X, dtype=np.float64)
        
        if self.fit_intercept:
            X = self._add_intercept(X)
        
        return X @ self.weights_
    
    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """
        Calculate R-squared (coefficient of determination).
        
        R^2 = 1 - SS_res / SS_tot
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Feature matrix
        y : ndarray of shape (n_samples,)
            True target values
        
        Returns
        -------
        r2 : float
            R-squared score
        """
        y = np.asarray(y, dtype=np.float64)
        y_pred = self.predict(X)
        
        # Total sum of squares
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        # Residual sum of squares
        ss_res = np.sum((y - y_pred) ** 2)
        
        # Handle edge case where ss_tot is zero
        if ss_tot == 0:
            return 0.0 if ss_res > 0 else 1.0
        
        return 1 - (ss_res / ss_tot)
    
    @property
    def intercept_(self) -> float:
        """Return the intercept term."""
        if self.weights_ is None:
            raise RuntimeError("Model must be fitted first")
        if self.fit_intercept:
            return self.weights_[0]
        return 0.0
    
    @property
    def coef_(self) -> np.ndarray:
        """Return the feature coefficients."""
        if self.weights_ is None:
            raise RuntimeError("Model must be fitted first")
        if self.fit_intercept:
            return self.weights_[1:]
        return self.weights_

### 2.1 Testing the Implementation

In [ ]:
# Generate synthetic data for testing
np.random.seed(42)
n_samples = 100
X_test = 2 * np.random.rand(n_samples, 1)
y_test = 4 + 3 * X_test.flatten() + np.random.randn(n_samples) * 0.5

print(f"True parameters: intercept=4, slope=3")
print(f"Data shape: X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Test closed-form solution
model_cf = LinearRegression(method='closed_form')
model_cf.fit(X_test, y_test)

print("Closed-Form Solution:")
print(f"  Intercept: {model_cf.intercept_:.4f}")
print(f"  Coefficient: {model_cf.coef_[0]:.4f}")
print(f"  R-squared: {model_cf.score(X_test, y_test):.4f}")

In [ ]:
# Test gradient descent solution
model_gd = LinearRegression(
    method='gradient_descent',
    learning_rate=0.1,
    n_iterations=1000,
    tolerance=1e-8
)
model_gd.fit(X_test, y_test)

print("\nGradient Descent Solution:")
print(f"  Intercept: {model_gd.intercept_:.4f}")
print(f"  Coefficient: {model_gd.coef_[0]:.4f}")
print(f"  R-squared: {model_gd.score(X_test, y_test):.4f}")

In [ ]:
# Visualize gradient descent convergence
plt.figure(figsize=(10, 4))
plt.plot(model_gd.cost_history_)
plt.xlabel('Iteration')
plt.ylabel('Cost (MSE/2)')
plt.title('Gradient Descent Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Training & Optimization

Using sklearn's diabetes dataset for a more realistic multivariate example.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

print(f"Dataset shape: {X.shape}")
print(f"Feature names: {diabetes.feature_names}")
print(f"Target range: [{y.min():.1f}, {y.max():.1f}]")

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Feature scaling for gradient descent
# Note: Closed-form doesn't require scaling, but GD benefits from it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train with closed-form solution (no scaling needed)
model_cf = LinearRegression(method='closed_form')
model_cf.fit(X_train, y_train)

print("Closed-Form Results:")
print(f"  Train R2: {model_cf.score(X_train, y_train):.4f}")
print(f"  Test R2:  {model_cf.score(X_test, y_test):.4f}")

In [ ]:
# Train with gradient descent (using scaled data)
model_gd = LinearRegression(
    method='gradient_descent',
    learning_rate=0.1,
    n_iterations=2000,
    tolerance=1e-8
)
model_gd.fit(X_train_scaled, y_train)

print("\nGradient Descent Results (scaled data):")
print(f"  Train R2: {model_gd.score(X_train_scaled, y_train):.4f}")
print(f"  Test R2:  {model_gd.score(X_test_scaled, y_test):.4f}")

In [ ]:
# Learning rate comparison
learning_rates = [0.001, 0.01, 0.1, 0.5]

plt.figure(figsize=(12, 4))

for lr in learning_rates:
    model = LinearRegression(
        method='gradient_descent',
        learning_rate=lr,
        n_iterations=500
    )
    model.fit(X_train_scaled, y_train)
    plt.plot(model.cost_history_, label=f'lr={lr}')

plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Effect of Learning Rate on Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Diagnostics & Evaluation

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Compute regression evaluation metrics.
    
    Parameters
    ----------
    y_true : array-like
        True target values
    y_pred : array-like
        Predicted values
    
    Returns
    -------
    dict : Dictionary containing MSE, RMSE, MAE, R2, and Adjusted R2
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    n = len(y_true)
    
    # Mean Squared Error
    mse = np.mean((y_true - y_pred) ** 2)
    
    # Root Mean Squared Error
    rmse = np.sqrt(mse)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(y_true - y_pred))
    
    # R-squared
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

In [ ]:
# Compute predictions and metrics
y_train_pred = model_cf.predict(X_train)
y_test_pred = model_cf.predict(X_test)

train_metrics = compute_metrics(y_train, y_train_pred)
test_metrics = compute_metrics(y_test, y_test_pred)

print("Training Metrics:")
for name, value in train_metrics.items():
    print(f"  {name}: {value:.4f}")

print("\nTest Metrics:")
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")

In [ ]:
# Residual analysis
residuals = y_test - y_test_pred

print("Residual Statistics:")
print(f"  Mean: {np.mean(residuals):.4f} (should be ~0)")
print(f"  Std:  {np.std(residuals):.4f}")
print(f"  Min:  {np.min(residuals):.4f}")
print(f"  Max:  {np.max(residuals):.4f}")

In [ ]:
def plot_learning_curves(model_class, X, y, train_sizes=None, cv=5, random_state=42):
    """
    Plot learning curves showing train/validation performance vs training set size.
    """
    if train_sizes is None:
        train_sizes = np.linspace(0.1, 1.0, 10)
    
    train_scores = []
    val_scores = []
    
    n_samples = len(y)
    indices = np.arange(n_samples)
    np.random.seed(random_state)
    np.random.shuffle(indices)
    
    # Use 20% for validation
    val_size = int(0.2 * n_samples)
    val_idx = indices[:val_size]
    train_idx = indices[val_size:]
    
    X_val, y_val = X[val_idx], y[val_idx]
    X_train_full, y_train_full = X[train_idx], y[train_idx]
    
    actual_sizes = []
    
    for size in train_sizes:
        n_train = int(size * len(X_train_full))
        actual_sizes.append(n_train)
        
        X_subset = X_train_full[:n_train]
        y_subset = y_train_full[:n_train]
        
        model = model_class(method='closed_form')
        model.fit(X_subset, y_subset)
        
        train_scores.append(model.score(X_subset, y_subset))
        val_scores.append(model.score(X_val, y_val))
    
    plt.figure(figsize=(10, 6))
    plt.plot(actual_sizes, train_scores, 'o-', label='Training score')
    plt.plot(actual_sizes, val_scores, 'o-', label='Validation score')
    plt.xlabel('Training set size')
    plt.ylabel('R2 Score')
    plt.title('Learning Curves')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return actual_sizes, train_scores, val_scores

# Plot learning curves
_ = plot_learning_curves(LinearRegression, X, y)

---
## 5. Visualizations

In [ ]:
def plot_regression_diagnostics(y_true, y_pred, figsize=(14, 10)):
    """
    Create comprehensive diagnostic plots for regression analysis.
    
    Plots:
    1. Predicted vs Actual
    2. Residuals vs Predicted
    3. Residual Distribution
    4. Q-Q Plot
    """
    residuals = y_true - y_pred
    
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    
    # 1. Predicted vs Actual
    ax1 = axes[0, 0]
    ax1.scatter(y_true, y_pred, alpha=0.6, edgecolors='k', linewidth=0.5)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect fit')
    ax1.set_xlabel('Actual Values')
    ax1.set_ylabel('Predicted Values')
    ax1.set_title('Predicted vs Actual')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Residuals vs Predicted (check homoscedasticity)
    ax2 = axes[0, 1]
    ax2.scatter(y_pred, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
    ax2.axhline(y=0, color='r', linestyle='--', lw=2)
    ax2.set_xlabel('Predicted Values')
    ax2.set_ylabel('Residuals')
    ax2.set_title('Residuals vs Predicted (Homoscedasticity Check)')
    ax2.grid(True, alpha=0.3)
    
    # 3. Residual Distribution
    ax3 = axes[1, 0]
    ax3.hist(residuals, bins=30, edgecolor='black', alpha=0.7, density=True)
    
    # Overlay normal distribution
    x_norm = np.linspace(residuals.min(), residuals.max(), 100)
    mu, sigma = np.mean(residuals), np.std(residuals)
    y_norm = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_norm - mu) / sigma) ** 2)
    ax3.plot(x_norm, y_norm, 'r-', lw=2, label=f'Normal(mu={mu:.1f}, sigma={sigma:.1f})')
    
    ax3.set_xlabel('Residuals')
    ax3.set_ylabel('Density')
    ax3.set_title('Residual Distribution')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Q-Q Plot (check normality)
    ax4 = axes[1, 1]
    
    # Calculate theoretical quantiles
    sorted_residuals = np.sort(residuals)
    n = len(sorted_residuals)
    theoretical_quantiles = np.array([(i - 0.5) / n for i in range(1, n + 1)])
    
    # Convert to standard normal quantiles
    from scipy import stats
    theoretical_quantiles = stats.norm.ppf(theoretical_quantiles)
    
    # Standardize residuals
    standardized_residuals = (sorted_residuals - np.mean(residuals)) / np.std(residuals)
    
    ax4.scatter(theoretical_quantiles, standardized_residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
    
    # Add reference line
    ax4.plot([-3, 3], [-3, 3], 'r--', lw=2, label='Reference line')
    
    ax4.set_xlabel('Theoretical Quantiles')
    ax4.set_ylabel('Sample Quantiles')
    ax4.set_title('Q-Q Plot (Normality Check)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(-3.5, 3.5)
    ax4.set_ylim(-3.5, 3.5)
    
    plt.tight_layout()
    plt.show()

# Create diagnostic plots
plot_regression_diagnostics(y_test, y_test_pred)

In [ ]:
# Feature importance visualization
def plot_feature_importance(model, feature_names, figsize=(10, 6)):
    """
    Plot feature coefficients as a bar chart.
    """
    coefs = model.coef_
    
    # Sort by absolute value
    sorted_idx = np.argsort(np.abs(coefs))[::-1]
    
    plt.figure(figsize=figsize)
    colors = ['green' if c > 0 else 'red' for c in coefs[sorted_idx]]
    plt.barh(range(len(coefs)), coefs[sorted_idx], color=colors, alpha=0.7)
    plt.yticks(range(len(coefs)), [feature_names[i] for i in sorted_idx])
    plt.xlabel('Coefficient Value')
    plt.title('Feature Importance (Coefficient Magnitude)')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

# Plot feature importance
plot_feature_importance(model_cf, diabetes.feature_names)

In [ ]:
# Simple regression visualization (using single feature)
def plot_simple_regression(X_feature, y, feature_name='Feature'):
    """
    Plot regression line for a single feature.
    """
    model = LinearRegression(method='closed_form')
    X_reshaped = X_feature.reshape(-1, 1)
    model.fit(X_reshaped, y)
    
    plt.figure(figsize=(10, 6))
    plt.scatter(X_feature, y, alpha=0.6, edgecolors='k', linewidth=0.5, label='Data points')
    
    # Create line for plotting
    X_line = np.linspace(X_feature.min(), X_feature.max(), 100).reshape(-1, 1)
    y_line = model.predict(X_line)
    
    plt.plot(X_line, y_line, 'r-', lw=2, 
             label=f'Regression line (y = {model.intercept_:.2f} + {model.coef_[0]:.2f}x)')
    
    plt.xlabel(feature_name)
    plt.ylabel('Target')
    plt.title(f'Linear Regression: {feature_name} vs Target')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return model

# Plot regression for the most important feature (bmi)
bmi_idx = list(diabetes.feature_names).index('bmi')
_ = plot_simple_regression(X[:, bmi_idx], y, 'BMI')

---
## 6. Use Cases & Guidelines

### 6.1 When to Use Linear Regression

**Appropriate Use Cases:**

1. **Linear Relationships**: When the relationship between features and target is approximately linear

2. **Baseline Model**: As a first model to establish a performance baseline before trying complex models

3. **Interpretability Required**: When you need to understand and explain the relationship between features and predictions

4. **Low-dimensional Data**: Works well when number of features is much smaller than number of samples

5. **Continuous Target Variable**: Predicting continuous outcomes (prices, temperatures, scores)

6. **Feature Importance Analysis**: Understanding which features contribute most to predictions

### 6.2 When NOT to Use Linear Regression

**Inappropriate Use Cases:**

1. **Non-linear Relationships**: When the true relationship is polynomial, exponential, or otherwise non-linear
   - *Alternative*: Polynomial regression, decision trees, neural networks

2. **Multicollinearity Present**: When features are highly correlated with each other
   - *Alternative*: Ridge/Lasso regression, PCA before regression

3. **Outliers**: Linear regression is sensitive to outliers
   - *Alternative*: Robust regression (Huber, RANSAC)

4. **Classification Problems**: Linear regression is for continuous targets
   - *Alternative*: Logistic regression, SVM, decision trees

5. **High-dimensional Data**: When $p > n$ (more features than samples)
   - *Alternative*: Regularized regression (Ridge, Lasso, Elastic Net)

6. **Heteroscedasticity**: When error variance changes with predicted value
   - *Alternative*: Weighted least squares, transform the target

### 6.3 Feature Scaling Importance

**For Closed-Form Solution:**
- Not strictly necessary (mathematically equivalent)
- Can help with numerical stability

**For Gradient Descent:**
- **Critical for convergence!**
- Features with larger scales dominate the gradient
- Without scaling, may need very small learning rates

In [ ]:
# Demonstration: Effect of scaling on gradient descent
print("WITHOUT Feature Scaling:")
model_unscaled = LinearRegression(
    method='gradient_descent',
    learning_rate=0.0001,  # Need very small learning rate
    n_iterations=1000
)
model_unscaled.fit(X_train, y_train)
print(f"  R2 (1000 iterations, lr=0.0001): {model_unscaled.score(X_train, y_train):.4f}")

print("\nWITH Feature Scaling:")
model_scaled = LinearRegression(
    method='gradient_descent',
    learning_rate=0.1,  # Can use larger learning rate
    n_iterations=1000
)
model_scaled.fit(X_train_scaled, y_train)
print(f"  R2 (1000 iterations, lr=0.1): {model_scaled.score(X_train_scaled, y_train):.4f}")

### 6.4 Pros and Cons Summary

| Pros | Cons |
|------|------|
| Simple and interpretable | Assumes linear relationship |
| Fast training and prediction | Sensitive to outliers |
| No hyperparameters (closed-form) | Can't capture complex patterns |
| Works well with limited data | Suffers from multicollinearity |
| Feature importance from coefficients | Requires feature engineering for non-linearity |
| Well-understood theory | Assumes homoscedasticity |

---
## 7. Comparison with sklearn

In [ ]:
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
import time

In [ ]:
# Train our implementation
start = time.time()
our_model = LinearRegression(method='closed_form')
our_model.fit(X_train, y_train)
our_time = time.time() - start

# Train sklearn implementation
start = time.time()
sklearn_model = SklearnLinearRegression()
sklearn_model.fit(X_train, y_train)
sklearn_time = time.time() - start

In [ ]:
# Compare coefficients
print("Coefficient Comparison:")
print("-" * 60)
print(f"{'Feature':<15} {'Our Model':>15} {'sklearn':>15} {'Difference':>12}")
print("-" * 60)

for i, name in enumerate(diabetes.feature_names):
    our_coef = our_model.coef_[i]
    sklearn_coef = sklearn_model.coef_[i]
    diff = abs(our_coef - sklearn_coef)
    print(f"{name:<15} {our_coef:>15.4f} {sklearn_coef:>15.4f} {diff:>12.2e}")

print("-" * 60)
print(f"{'Intercept':<15} {our_model.intercept_:>15.4f} {sklearn_model.intercept_:>15.4f} {abs(our_model.intercept_ - sklearn_model.intercept_):>12.2e}")

In [ ]:
# Compare performance metrics
print("\nPerformance Comparison:")
print("-" * 50)

our_train_r2 = our_model.score(X_train, y_train)
our_test_r2 = our_model.score(X_test, y_test)

sklearn_train_r2 = sklearn_model.score(X_train, y_train)
sklearn_test_r2 = sklearn_model.score(X_test, y_test)

print(f"{'Metric':<20} {'Our Model':>12} {'sklearn':>12}")
print("-" * 50)
print(f"{'Train R2':<20} {our_train_r2:>12.6f} {sklearn_train_r2:>12.6f}")
print(f"{'Test R2':<20} {our_test_r2:>12.6f} {sklearn_test_r2:>12.6f}")
print(f"{'Training Time (s)':<20} {our_time:>12.6f} {sklearn_time:>12.6f}")

In [ ]:
# Compare predictions
our_preds = our_model.predict(X_test)
sklearn_preds = sklearn_model.predict(X_test)

pred_diff = np.abs(our_preds - sklearn_preds)

print("\nPrediction Comparison:")
print(f"  Max absolute difference: {pred_diff.max():.2e}")
print(f"  Mean absolute difference: {pred_diff.mean():.2e}")
print(f"  Predictions are virtually identical: {pred_diff.max() < 1e-10}")

In [ ]:
# Visual comparison of predictions
plt.figure(figsize=(12, 5))

# Plot 1: Predictions comparison
plt.subplot(1, 2, 1)
plt.scatter(sklearn_preds, our_preds, alpha=0.6, edgecolors='k', linewidth=0.5)
plt.plot([sklearn_preds.min(), sklearn_preds.max()], 
         [sklearn_preds.min(), sklearn_preds.max()], 'r--', lw=2)
plt.xlabel('sklearn Predictions')
plt.ylabel('Our Model Predictions')
plt.title('Prediction Comparison')
plt.grid(True, alpha=0.3)

# Plot 2: Coefficient comparison
plt.subplot(1, 2, 2)
x_pos = np.arange(len(diabetes.feature_names))
width = 0.35
plt.bar(x_pos - width/2, our_model.coef_, width, label='Our Model', alpha=0.7)
plt.bar(x_pos + width/2, sklearn_model.coef_, width, label='sklearn', alpha=0.7)
plt.xticks(x_pos, diabetes.feature_names, rotation=45, ha='right')
plt.xlabel('Feature')
plt.ylabel('Coefficient')
plt.title('Coefficient Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.1 Summary

Our implementation produces results virtually identical to sklearn's LinearRegression. Key observations:

1. **Coefficients**: Match to machine precision (~1e-10 difference)
2. **Predictions**: Identical within floating-point tolerance
3. **Performance**: sklearn may be slightly faster due to optimized BLAS libraries

**When to use our implementation:**
- Learning purposes
- Understanding the mathematics
- Custom modifications needed

**When to use sklearn:**
- Production code
- Need additional features (regularization available via Ridge/Lasso)
- Integration with sklearn pipelines

---
## Conclusion

This notebook covered:

1. **Theory**: OLS derivation, normal equation, gradient descent, and key assumptions
2. **Implementation**: Built LinearRegression from scratch with both closed-form and gradient descent
3. **Training**: Applied to the diabetes dataset with proper scaling
4. **Diagnostics**: Computed MSE, RMSE, R2, and analyzed residuals
5. **Visualizations**: Created regression plots, residual analysis, Q-Q plots
6. **Guidelines**: When to use/avoid linear regression, importance of scaling
7. **Comparison**: Validated against sklearn implementation

Linear regression remains a fundamental tool in machine learning, serving as both a practical model for linear relationships and a stepping stone to more complex algorithms.